# Fine Tuning A Chatbot for Medical questions:
Developing a medical chat system using medical Q and A data.

## Part 2: Training
This noebook was run via Google colab using an A100 GPU and high-ram. Training an LLM takes more resources than my personal set up can provide.

### 2a: Load data, split data into training and testing

In [ ]:
# Install dependencies into our runtime
!pip install -q transformers datasets accelerate bitsandbytes peft trl


import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from peft import LoraConfig
from trl import SFTTrainer

# Load the dataset (was uploaded as out.jsonl to colab)
dataset = load_dataset("json", data_files="out.jsonl")

# Split train/test (90/10)
dataset = dataset["train"].train_test_split(test_size=0.1, seed=42)

In [ ]:
# Define hugging face token that will allow us to use their models
# Note that it is free to define a token associated with your huggingface account.
# You will also need to accept the terms and conditions for the model(s) you want to use.
TOKEN = <redacted>

# I have removed my token from this notebook. I recommend setting your environment variables up in your colab settings.
# Next time, I would do that instead.

### 2b: Define our model, LoRA, and train

In [ ]:
# Define the model and tokenizer.
# We will use_fast, which is an efficiency option useful in batching.
MODEL = "meta-llama/Meta-Llama-3.1-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL, use_fast=True, token=TOKEN)
tokenizer.pad_token = tokenizer.eos_token  

# Use the huggingface function to load our pre-trained base model.
# This function ensures we have the correct weights and configuration for the model
# we want to use.

# Note our parameters:
## - Load_in_4bit= True: compress the weights into a 4-bit format, useful to save memory since we have limited resources
## - device_map="auto": automatically determine the optimal way to distribute our model layers across our GPUs
## - torch_dtype=torch.float16: which datatype the weights should be loaded in. I used a smaller size to help with memory constraints
model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    load_in_4bit=True,
    device_map="auto",
    torch_dtype=torch.float16,
    token=TOKEN
)

# Define our LoRA configuration
# According to Hugging face:
# > Low-Rank Adaptation (LoRA) is a PEFT method that decomposes a large matrix into two smaller low-rank matrices in the attention layers. 
#> This drastically reduces the number of parameters that need to be fine-tuned.

# I'm mostly using LoRA to speed up the training time for the fine-runing and prevent memory issues.
# I originally tried a larger config with r=16, lora_alpha=16 and ran into errors related to RAM.
 
# Note the parameters I landed on:
## - r=8: this is our rank. It determines the size of the matrices we are using to adapt our pretrainedmodel. 
##      While a low rank like 8 may not get us the best results, our task is not particularly complex so I am optimistic.
## - lora_alpha=16: our scaling, to tell the model to rely more heavily on the newly learned weights. If I had more resources 
##      and time, I would try larger options here in another attempt.
## - lora_dropout=0.05: a regularization parameter, this prevents overfitting by randomly setting some of the lora activations
##      to zero during training. If the model appears to be overfitting, we would want to increase this rate on our next attempt.
## - bias="none": recommended for memory efficiency, we will not updte the biases during training
## - task_type="CAUSAL_LM": We are specifying causal language modeling type, which is commonly used in chat applications. We are
##      We will be predicting tokens in a sequence, and the model will not know future tokens.

peft_config = LoraConfig(
    r=8,
    lora_alpha=16,   
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Define our training arguments
# Parameters:
## - per_device_train_batch_size=1: used to prevent my CUDA out of memory errors, small batch reduces memory used. If
##      if i had time to do this again, I would try a larger batch. It would probably be faster. I think I overcorrectly
##      after running into so many memory issues.
## - per_device_eval_batch_size=1: Same here, keeping things small.
## - gradient_accumulation_steps=16: Lets us train on more data then would fit in memory.
## - num_train_epochs=3: Number of passes through the data in training. I settled on 3, because it allows for iterative
##      learning of the data but wouldn't take too long. If I had more time and wanted to improve my results, I would try 5-7.
## - learning_rate=2e-4: the size of steps we take through the gradient as we update our weights. With finetuning, it's important for 
##      this number to be small so that we don't accidentally overfit to the pre-trained knowledge.
## - fp16=True: Enable mixed precision training and use 16 bits to store our numbers. This is again a smaller option to prevent memory issues.
## - logging_steps=25: how often I want to see logs during training.
## - save_strategy="steps": Save our progress during training. Important incaseour colab runtime is unstable.
## - save_steps=200: How often to save.
## - eval_strategy="steps": Evaluate our progress against the test set after a certain number of steps. I choose this option to
##  monitor the progress of our training.
## - eval_steps=200: We will evaluate every 200 steps

training_args = TrainingArguments(
    output_dir="./llama-lora-finetuned",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=25,
    save_strategy="steps",
    save_steps=200,
    eval_strategy="steps",
    eval_steps=200,
    save_total_limit=2,
    push_to_hub=False,
    report_to="none"
)

# Define our trainer
## Use the previously defined training and testing data, the tokenizer that goes with our pretrained model,
## and the args and LoRA defined above.
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    processing_class=tokenizer,
    args=training_args,
    peft_config=peft_config,
)

#  Train 
trainer.train()

# Save our model
trainer.save_model("./llama-lora-finetuned")
tokenizer.save_pretrained("./llama-lora-finetuned")


## * NOTE: I woulld break this cell up into multiple steps to be easier to read if my resources weren't so limited.
## I had a lot of issues with memory constraints and it was easier for me to kick off the whole process at once upon failure.

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

...

Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
200,0.924700,0.903941,0.948113,985432.000000,0.769559


/usr/local/lib/python3.12/dist-packages/peft/utils/other.py:1228: UserWarning: Unable to fetch remote file due to the following error 401 Client Error. (Request ID: Root=1-68d87f7f-04a0074362036ef1643c43ce;772a4e24-a84e-4a6b-b101-1bd6c98f9818)

Cannot access gated repo for url https://huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct/resolve/main/config.json.
Access to model meta-llama/Llama-3.1-8B-Instruct is restricted. You must have access to it and be authenticated to access it. Please log in. - silently ignoring the lookup for the file config.json in meta-llama/Meta-Llama-3.1-8B-Instruct.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:286: UserWarning: Could not find a config file in meta-llama/Meta-Llama-3.1-8B-Instruct - will assume that the vocabulary was not modified.
  warnings.warn(


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
200,0.924700,0.903941,0.948113,985432.000000,0.769559
400,0.889200,0.871998,0.905122,1936339.000000,0.774813
600,0.945200,0.856448,0.855901,2891110.000000,0.778287
800,0.910100,0.845035,0.832319,3857544.000000,0.780248
1000,0.881400,0.839191,0.822204,4820198.000000,0.781377
1200,0.856500,0.831792,0.815052,5787241.000000,0.784255
1400,0.851800,0.826820,0.840711,6761069.000000,0.785303
1600,0.869500,0.822905,0.809057,7725755.000000,0.786112
1800,0.839300,0.818628,0.815307,8688462.000000,0.787008
2000,0.827300,0.817720,0.809123,9645684.000000,0.786797


/usr/local/lib/python3.12/dist-packages/peft/utils/other.py:1228: UserWarning: Unable to fetch remote file due to the following error 401 Client Error. (Request ID: Root=1-68d883b2-336f28200110482813327e0f;0858f7c6-ba45-4458-abbb-216aff186a39)

Cannot access gated repo for url https://huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct/resolve/main/config.json.
Access to model meta-llama/Llama-3.1-8B-Instruct is restricted. You must have access to it and be authenticated to access it. Please log in. - silently ignoring the lookup for the file config.json in meta-llama/Meta-Llama-3.1-8B-Instruct.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:286: UserWarning: Could not find a config file in meta-llama/Meta-Llama-3.1-8B-Instruct - will assume that the vocabulary was not modified.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/other.py:1228: UserWarning: Unable to fetch remote file due to the following error 401 Client Error. (R

('./llama-lora-finetuned/tokenizer_config.json',
 './llama-lora-finetuned/special_tokens_map.json',
 './llama-lora-finetuned/chat_template.jinja',
 './llama-lora-finetuned/tokenizer.json')

### 2c: Save everything
Before doing anything else, save and download the model artifacts.  Google colab is prone to runtime restarts and the data will not persist. The training took 4 hours and I would hate to have to run it again.

Note that in the previous section I had:
```
trainer.save_model("./llama-lora-finetuned")
tokenizer.save_pretrained("./llama-lora-finetuned")
```



In [ ]:
# Download our model

import shutil
from google.colab import files

shutil.make_archive("llama-lora-finetuned", 'zip', "./llama-lora-finetuned")

files.download("llama-lora-finetuned.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### 2d: Evluate
Use the Trainer's built in evaluation for some initial metrics. Note that evaluation for generative tasks is not straight forward. I will provide a more in depth evalution in Part3.

In [ ]:
trainer.evaluate()

{'eval_loss': 0.8120278120040894,
 'eval_runtime': 214.053,
 'eval_samples_per_second': 7.666,
 'eval_steps_per_second': 7.666,
 'eval_entropy': 0.7991135515894634,
 'eval_num_tokens': 13344000.0,
 'eval_mean_token_accuracy': 0.7887991961366443,
 'epoch': 3.0}

### 2e: Quick check
Can the model answer medical questions? More on this in Part3.

In [ ]:
from transformers import pipeline

# Format a message the same way we had in our training data.
messages = [{"role": "user", "content": "What is a stroke?"}]

# Use the chat template to ensure our new input matches our tokenizer
prompt = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)

# Set up a pipeline for output generation
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    torch_dtype=torch.float16,
    device_map="auto",
)
# Grab our output
outputs = pipe(prompt, max_new_tokens=120, do_sample=True)

print(outputs[0]["generated_text"])

`torch_dtype` is deprecated! Use `dtype` instead!
Device set to use cuda:0


<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

<|eot_id|><|start_header_id|>user<|end_header_id|>

What is a stroke?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

A stroke is an attack on the brain. It happens when the flow of oxygen-rich blood to a part of the brain is blocked. Without this blood flow, brain cells quickly become damaged or die. The part of the body controlled by the damaged part of the brain also becomes damaged. The damage can result in loss of memory, movement, or sensation and can affect a person's ability to think, walk, talk, and control their bowel and bladder. It can also cause numbness or weakness in the face, arm, or leg. A stroke is a medical emergency. Prompt treatment is crucial to


In [12]:
# Triple check I've saved everything
model.save_pretrained("llama-finetuned-medchat")
tokenizer.save_pretrained("llama-finetuned-medchat")

('llama-finetuned-medchat/tokenizer_config.json',
 'llama-finetuned-medchat/special_tokens_map.json',
 'llama-finetuned-medchat/chat_template.jinja',
 'llama-finetuned-medchat/tokenizer.json')

In [ ]:
import shutil
shutil.make_archive("llama-finetuned-medchat", 'zip', "./llama-finetuned-medchat")

from google.colab import files
files.download("llama-finetuned-medchat")

### 2f: Upload my model to hugging face!
My model is here: https://huggingface.co/sarahgc/llama-lora-medchat

In [14]:
from huggingface_hub import HfApi

api = HfApi(token="hf_YOUR_ACCESS_TOKEN")


api.upload_folder(
    folder_path="llama-finetuned-medchat",
    repo_id="sarahgc/llama-lora-medchat",
    repo_type="model",
)

RepositoryNotFoundError: 401 Client Error. (Request ID: Root=1-68d8b9b7-44941ba6512bc38d12ef94ee;21e4fc03-27e6-4d69-a472-becd5985a0b0)

Repository Not Found for url: https://huggingface.co/api/models/sarahgc/llama-lora-medchat/preupload/main.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a private or gated repo, make sure you are authenticated. For more details, see https://huggingface.co/docs/huggingface_hub/authentication
Invalid username or password.
Note: Creating a commit assumes that the repo already exists on the Huggingface Hub. Please use `create_repo` if it's not the case.